# 09b: Main results for LightGBM and random forest (E1)

Design decisions
- The frozen design of each task is kept: training portion and seed 42, the 2,000 prior training patients of the frozen prior and lambda, M, the feature-order seed and search budget, the explained subsamples of calibration split 2 and of the test split, calibration split 1 for certificates, the target grids (alpha_0 0.02, 0.05, 0.10; beta 0.2, 0.3, 0.5) and the frozen alpha_sel. Only the model changes.
- Settings closest to the frozen XGBoost, as in notebook 12b: LightGBM with 100 trees of depth 4 (15 leaves), 32 bins, learning rate 0.3, at least 50 records per leaf and minimum split gain 2; random forest with the plan's 50 trees of depth 4 and at least 50 records per leaf. Each model is fitted on the training portion with a Youden threshold on 5-fold out-of-fold scores, exported, checked against the native library on every training row, saved and hashed before calibration split 1 is read.
- Every box that could be released at the loosest grid target is re-checked by the second exact check; release, the policy test on calibration split 2, the test results and the consistency check follow notebook 09. The XGBoost rows are those of results/09_main.csv.
- Each task and family is saved on Drive when finished, and candidate sets every 500 new cells, so an interrupted run resumes. Expected run time on Colab CPU: about 4 to 6 hours in all (candidate sets for about 52,000 patients); it can be stopped and run again at any point.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public, splits
from scax.experiments.main_results import FrozenTask
from scax.experiments import model_ablations as ma
from scax.experiments import shift as sh
from scax.metrics.model_metrics import model_metrics
from scax.models.export import Ensemble
try:
    from scax.experiments import families as fam
except ImportError:
    raise RuntimeError('The stage 12 source is not loaded. Run the stage 12 batch, then Runtime > Disconnect and delete runtime, then run again.')

SEED = 42
seed_everything(SEED)
TASKS = ['cdc_diabetes', 'brfss2015_heart']
FAMILIES = ['lightgbm', 'sklearn_rf']
NAMES = {'cdc_diabetes': 'CDC Diabetes', 'brfss2015_heart': 'BRFSS Heart disease'}
LIMIT = None          # every frozen explained patient and prior patient
REC05 = json.load(open(f'{paths.RESULTS}/05_frozen.json'))
assert REC05['written_before_calibration_split_1']
FT = {t: FrozenTask(t, REPO_DIR, DRIVE_ROOT, REC05['frozen_config_sha256'][t]) for t in TASKS}
OUT_DIR = paths.DERIVED / 'families'; OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG = lambda m: print(m, flush=True)
DATA, PRIOR_ROWS = {}, {}
for t in TASKS:
    X, y = public.LOADERS[t](paths.RAW)
    code = np.load(paths.DERIVED / 'splits' / f'{t}.npy')
    assert splits.summary(code)['sha256'] == FT[t].cfg['split_sha256'], 'split differs from the frozen one'
    DATA[t] = (X.to_numpy(dtype=float), y, code, list(X.columns))
    PRIOR_ROWS[t] = np.array(json.load(open(DRIVE_ROOT + '/' + FT[t].cfg['prior']['file']))['training_rows'])
    assert len(PRIOR_ROWS[t]) == FT[t].cfg['prior']['training_patients']
    print(t, 'frozen configuration', FT[t].cfg_sha[:16], '| targets', {k: FT[t].targets[k] for k in ('alpha_0', 'beta', 'alpha_sel')},
          '| explained: calibration split 2', len(FT[t].cal2_rows), 'test', len(FT[t].test_rows))

In [ ]:
def lim(a):
    return a if LIMIT is None else a[:LIMIT]


def run_unit(task, family):
    """One task and model family on the frozen design; saved on Drive when finished."""
    res_file = OUT_DIR / f'{task}__{family}__results.json'
    if res_file.exists():
        print(f'{task} | {family}: loaded from Drive', flush=True)
        return json.load(open(res_file))
    t0 = time.time()
    ft = FT[task]; T = ft.targets; Xa, y, code, names = DATA[task]
    tr = splits.indices(code, 'train')
    params = fam.family_params(ft.cfg['model']['params'], names, family)
    model, ens, pred = ma.fit_variant(family, Xa[tr], y[tr], params, SEED, names)
    model_file = OUT_DIR / f'{task}__{family}__model.json'
    if model_file.exists():
        assert Ensemble.load(model_file).hash() == ens.hash(), f'{task} {family}: refitting does not reproduce the saved model'
    else:
        ens.save(model_file)
    agree = ma.exported_vs_native_detail(model, family, pred, Xa[tr])
    assert agree['beyond_rounding'] == 0, f'{task} {family}: exported and native models disagree: {agree}'
    c = ft.cfg['candidates']
    sm = sh.ShiftModel(f'{task} {family}', pred, c['M'], c['order_seed'], c['node_budget'], ft.lam,
                       {0: T['alpha_0'], 1: T['alpha_1']}, T['delta_1'], cache_path=OUT_DIR / f'{task}__{family}__candidates.pkl')
    sm.set_prior(Xa[lim(PRIOR_ROWS[task])], log=LOG)
    write_json(OUT_DIR / f'{task}__{family}__before_calibration.json', {
        'model_sha256': ens.hash(), 'tau': float(ens.tau), 'settings': params, 'exported_vs_native_training': agree,
        'prior_boxes': len(sm.counts), 'lambda': ft.lam})          # fixed before calibration split 1 is read
    cal1 = splits.indices(code, 'cal1')
    sm.set_calibration(Xa[cal1], y[cal1])
    c2, te = lim(ft.cal2_rows), lim(ft.test_rows)
    run = fam.FamilyRun(sm, T, Xa[c2], y[c2], Xa[te], y[te], log=LOG)
    rows, main = run.grid_rows()
    test_all = splits.indices(code, 'test')
    cons = run.consistency(main, sm.g.cells(Xa[test_all]), y[test_all])
    p = model.predict_proba(Xa[test_all])[:, 1]
    mm = model_metrics(p, y[test_all], pred.predict(Xa[test_all]))
    r = {'task': task, 'family': family, 'settings': params, 'model_sha256': ens.hash(), 'tau': float(ens.tau),
         'exported_vs_native_training': agree, 'thresholds': int(sum(len(t) for t in sm.g.thresholds)),
         'prior_boxes': len(sm.counts), 'boxes': len(run.keys), 'eligible_boxes': run.eligible,
         'unverified_boxes': run.unverified, 'rows': rows, 'consistency': cons, 'model_metrics': mm,
         'median_seconds_per_cell': float(np.median(list(sm.gen.seconds.values()))), 'seconds': time.time() - t0}
    write_json(res_file, r)
    fr = [x for x in rows if x['frozen_targets']]
    print(f"{task} | {family}: AUROC {mm['auroc']:.3f}; " + '; '.join(
        f"{x['kind']}: {x['test_certified_share']:.1%} of test patients, error {x['test_error_rate']:.1%}, "
        f"policy {'certified' if x['policy_certified'] else 'not certified'} (U_sel {x['U_sel']:.3f} against {x['alpha_sel']})"
        for x in fr) + f"; {time.time() - t0:.0f}s", flush=True)
    return r

## LightGBM and random forest on both tasks

In [ ]:
UNITS = {}
for task in TASKS:
    for family in FAMILIES:
        UNITS[(task, family)] = run_unit(task, family)

## Results, with the frozen XGBoost rows of notebook 09

In [ ]:
MAIN = pd.DataFrame([{'task': t, 'family': f, **row} for (t, f), r in UNITS.items() for row in r['rows']])
X09 = pd.read_csv(f'{paths.RESULTS}/09_main.csv').assign(family='xgboost')
ALL = pd.concat([X09, MAIN], ignore_index=True)
ALL['family_name'] = ALL.family.map(fam.FAMILY_NAMES)
METRICS = pd.DataFrame([{'task': t, 'family': f, **r['model_metrics']} for (t, f), r in UNITS.items()])
M09 = pd.read_csv(f'{paths.RESULTS}/09_model_metrics.csv').assign(family='xgboost')
METRICS = pd.concat([M09, METRICS], ignore_index=True)
CONS = pd.DataFrame([{'task': t, 'family': f, **c} for (t, f), r in UNITS.items() for c in r['consistency']])
pd.set_option('display.width', 250)
COLS = ['task', 'family_name', 'kind', 'test_certified_share', 'test_certified_share_of_predicted_class', 'test_error_rate',
        'test_error_upper_95', 'mean_size', 'median_calibration_support', 'cal2_accepted', 'U_sel', 'alpha_sel', 'policy_certified']
print('Frozen targets (alpha_0 = 0.05, beta = 0.3):')
print(ALL[ALL.frozen_targets == True][COLS].sort_values(['task', 'kind', 'family_name']).round(4).to_string(index=False))
print('\nUnderlying models on the full test split:')
print(METRICS[['task', 'family', 'auroc', 'auprc', 'brier', 'ece_15_equal_mass', 'sensitivity', 'specificity']].round(4).to_string(index=False))
print('\nConsistency check (boxes released at the frozen targets; flagged = corrected lower limit above the target):')
print(CONS.groupby(['task', 'family', 'box_class']).agg(released_boxes=('test_n', 'size'), mean_test_error=('test_error', 'mean'),
                                                         flagged=('estimate_above_target', 'sum')).to_string())
print('\nboxes, eligible at the loosest target, not confirmed by the second check:')
print(pd.DataFrame([{'task': t, 'family': f, 'thresholds': r['thresholds'], 'boxes': r['boxes'], 'eligible': r['eligible_boxes'],
                     'unverified': r['unverified_boxes'], 'median_seconds_per_cell': r['median_seconds_per_cell'],
                     'hours': r['seconds'] / 3600} for (t, f), r in UNITS.items()]).round(3).to_string(index=False))

## Figure: certified share against the target, by model family

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})
STYLE = {'xgboost': ('#0072B2', 'o'), 'lightgbm': ('#D55E00', 's'), 'sklearn_rf': ('#009E73', '^')}
fig, axes = plt.subplots(2, 2, figsize=(7.4, 5.2))
for i, task in enumerate(TASKS):
    T = FT[task].targets
    for fmy, (col, mk) in STYLE.items():
        d = ALL[(ALL.task == task) & (ALL.family == fmy)]
        neg = d[(d.kind == 'negative outcome') & (d.beta == T['beta'])].sort_values('alpha_0')
        enr = d[(d.kind == 'enrichment') & (d.alpha_0 == T['alpha_0'])].sort_values('beta')
        axes[i, 0].plot(neg.alpha_0, neg.test_certified_share, color=col, marker=mk, ms=3.5, lw=1.4, label=fam.FAMILY_NAMES[fmy])
        axes[i, 1].plot(enr.beta, enr.test_certified_share, color=col, marker=mk, ms=3.5, lw=1.4, label=fam.FAMILY_NAMES[fmy])
    axes[i, 0].set_ylabel(f'{NAMES[task]}:\nshare of test patients')
    for ax in axes[i]:
        ax.set_ylim(bottom=0)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        ax.grid(color='0.92', lw=0.6)
axes[0, 0].set_title('Certified negative-outcome explanations', fontsize=8.5)
axes[0, 1].set_title('Certified enrichment explanations', fontsize=8.5)
axes[1, 0].set_xlabel(r'Target error for negative-outcome claims ($\alpha_0$)')
axes[1, 1].set_xlabel(r'Claimed minimum share with the outcome ($\beta$)')
for ax in axes[:, 0]:
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
for ax in axes[:, 1]:
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=3, frameon=False, fontsize=7.5, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(rect=(0, 0, 1, 0.95))
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_certified_share_by_family.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
ALL.to_csv(f'{paths.RESULTS}/09b_main.csv', index=False)
METRICS.to_csv(f'{paths.RESULTS}/09b_model_metrics.csv', index=False)
CONS.to_csv(f'{paths.RESULTS}/09b_box_consistency.csv', index=False)
write_json(f'{paths.RESULTS}/09b_families.json', {
    'seed': SEED, 'frozen_config_sha256': {t: FT[t].cfg_sha for t in TASKS}, 'limit': LIMIT,
    'units': {f'{t}__{f}': {k: v for k, v in r.items() if k not in ('rows', 'consistency')} for (t, f), r in UNITS.items()},
    'policy': {f'{t}__{f}': {x['kind']: {'certified': x['policy_certified'], 'U_sel': x['U_sel'], 'alpha_sel': x['alpha_sel'],
                                         'accepted': x['cal2_accepted']} for x in r['rows'] if x['frozen_targets']}
               for (t, f), r in UNITS.items()}})
print('written: results/09b_main.csv, 09b_model_metrics.csv, 09b_box_consistency.csv, 09b_families.json; figures/fig_certified_share_by_family')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "09b: main results for LightGBM and random forest on the frozen design (tasks B and A), policy test, consistency check, figure"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)